# 🥁 Drum OMR — Qwen3-VL fine-tune (trial)

Teaches a small vision-language model to read one bar of drum notation and write it as text:

```
0:eighth:hi_hat_closed+kick 4:eighth:hi_hat_closed 8:eighth:hi_hat_closed+snare ...
```

Same bars and same song split as the MobileNetV3 baseline (`baseline-14drum-v1`, 12.5%
of held-out bars read perfectly), so the final number is directly comparable.

**Before running**
1. On the Mac: `python3 ml/omr/prepare_finetune.py --zip` (add `--train-bars 3000` for a short trial)
2. Upload `ml/data/finetune.zip` to `MyDrive/unsloth/finetune.zip`
3. Runtime → Change runtime type → **T4 GPU**, then Run all

If Colab disconnects mid-training, run all again: training resumes from the last saved step.
A run that already finished is never retrained: its adapter is loaded and only missing scores are produced.

**First trial (2026-10-05):** 3,000 bars, 2 epochs, 133 min on a T4 → 72.4% of 312 held-out bars
read perfectly, against 12.5% for MobileNetV3.

In [ ]:
# 1. Install Unsloth (LoRA fine-tuning for vision models)
%pip install -q unsloth

In [ ]:
# 2. Settings, Drive, and the data package
from google.colab import drive
drive.mount('/content/drive')

import json, sys, zipfile
from pathlib import Path

# To compare models fairly, change only MODEL and keep everything else below the same.
# Tried: Qwen3-VL 8B (72.4%). Candidates, names not all checked against Hugging Face:
#   'unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit'   'unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit'
#   'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'         'unsloth/gemma-3-4b-it-unsloth-bnb-4bit'
MODEL = 'unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit'
# Free T4 settings. On a paid GPU with 40 GB or more, for the best accuracy use
# LOAD_IN_4BIT = False (with a full-precision model name such as 'Qwen/Qwen3-VL-8B-Instruct'),
# LORA_RANK = 64 and EVAL_BARS = None. A 32B model at full precision needs about 80 GB.
LOAD_IN_4BIT = True
LORA_RANK = 16
TRAIN_BARS = 3000    # bars spread evenly over all songs; None trains on every bar in the zip
APP_STYLE_BARS = 3000  # extra training bars redrawn in DrumHub's own notation; 0 for none, None for all
EPOCHS = 2           # use 1 with all ~15,800 bars (about 6 hours per pass on a T4 for the 8B)
# Folder of a finished run to load instead of training, e.g. the first trial:
# 'Qwen3-VL-8B-Instruct-unsloth-bnb-4bit'. None trains (or resumes) the run described above.
REUSE_RUN = None
IMAGE_SCALE = 2      # bars are only ~356x146 px; enlarging gives the model more to see per notehead
EVAL_BARS = 300      # evenly spread test bars to score; None scores all 2,184 (slow)
MAX_NEW_TOKENS = 512 # the longest bar in the data is ~700 characters

DRIVE_ROOT = Path('/content/drive/MyDrive/unsloth')

ROOT = Path('/content/drumhub')
# Extract again whenever the zip on Drive is a different size from the one already unpacked.
zip_size = str((DRIVE_ROOT / 'finetune.zip').stat().st_size)
marker = ROOT / '.complete'
if not marker.exists() or marker.read_text() != zip_size:
    with zipfile.ZipFile(DRIVE_ROOT / 'finetune.zip') as archive:
        archive.extractall(ROOT)
    marker.write_text(zip_size)

sys.path.insert(0, str(ROOT / 'ml' / 'omr'))
from prepare_finetune import decode_text  # the same parser the app side uses

PROMPT = (ROOT / 'ml/data/finetune/prompt.txt').read_text().strip()
def count_lines(name):
    path = ROOT / 'ml/data/finetune' / name
    return sum(1 for _ in path.open()) if path.exists() else 0


all_bars, all_app_bars = count_lines('train.jsonl'), count_lines('train_app.jsonl')
train_bars = min(TRAIN_BARS or all_bars, all_bars)
app_bars = all_app_bars if APP_STYLE_BARS is None else min(APP_STYLE_BARS, all_app_bars)
# Each model, training-set size and epoch count gets its own folder, so a new run never
# resumes another run's checkpoints.
RUN_DIR = DRIVE_ROOT / 'qwen-omr' / (REUSE_RUN or f"{MODEL.split('/')[-1]}-{train_bars}bars-{EPOCHS}ep-r{LORA_RANK}{'' if LOAD_IN_4BIT else '-16bit'}{f'-app{app_bars}' if app_bars else ''}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRAINED = (RUN_DIR / 'lora').exists()
print(f"{'Scoring the finished run in' if TRAINED else f'Training on {train_bars} + {app_bars} app-style bars into'} {RUN_DIR}")

In [ ]:
# 3. Load the image -> text pairs
from PIL import Image


def read_split(name):
    with (ROOT / 'ml/data/finetune' / f'{name}.jsonl').open() as source:
        return [json.loads(line) for line in source]


def load_image(path, scale=IMAGE_SCALE):
    # Kept grayscale: 15,800 colour images would not fit in Colab's memory.
    image = Image.open(ROOT / path).convert('L')
    return image.resize((image.width * scale, image.height * scale), Image.LANCZOS)


def to_conversation(row):
    return {'messages': [
        {'role': 'user', 'content': [{'type': 'text', 'text': PROMPT},
                                     {'type': 'image', 'image': load_image(row['image'], row.get('scale', IMAGE_SCALE))}]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': row['target']}]},
    ]}


train_rows, test_rows = read_split('train'), read_split('test')
train_rows = train_rows[::len(train_rows) // train_bars][:train_bars]
if app_bars:
    # App-drawn bars are 660x360, already about the size of an enlarged training bar.
    app_rows = read_split('train_app')
    train_rows += [{**row, 'scale': 1} for row in app_rows[::len(app_rows) // app_bars][:app_bars]]
train_data = [] if TRAINED else [to_conversation(row) for row in train_rows]
print(f'Train bars: {len(train_rows)} ({app_bars} app-style)   Test bars: {len(test_rows)}')
print(train_rows[10]['target'])
load_image(train_rows[10]['image'])

In [ ]:
# 4. Load Qwen3-VL in 4-bit and attach a LoRA adapter
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    str(RUN_DIR / 'lora') if TRAINED else MODEL, load_in_4bit=LOAD_IN_4BIT, use_gradient_checkpointing='unsloth')

# Vision layers are trained too: telling toms apart is a seeing problem, not a wording one.
if not TRAINED:
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=True, finetune_language_layers=True,
        finetune_attention_modules=True, finetune_mlp_modules=True,
        r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0, bias='none', random_state=42)

In [ ]:
# 5. Train. Checkpoints go to Drive so a disconnect does not lose the run.
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

if TRAINED:
    print('Already trained; skipping.')
else:
    FastVisionModel.for_training(model)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=train_data,
        args=SFTConfig(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_train_epochs=EPOCHS,
            learning_rate=2e-4,
            warmup_ratio=0.03,
            lr_scheduler_type='cosine',
            optim='adamw_8bit',
            weight_decay=0.01,
            logging_steps=10,
            save_strategy='steps', save_steps=100, save_total_limit=2,
            output_dir=str(RUN_DIR / 'checkpoints'),
            seed=42,
            report_to='none',
            # Required for vision fine-tuning:
            remove_unused_columns=False,
            dataset_text_field='',
            dataset_kwargs={'skip_prepare_dataset': True},
            max_length=2048,
        ),
    )
    resume = any((RUN_DIR / 'checkpoints').glob('checkpoint-*'))
    stats = trainer.train(resume_from_checkpoint=resume)
    model.save_pretrained(str(RUN_DIR / 'lora'))
    tokenizer.save_pretrained(str(RUN_DIR / 'lora'))
    print(f"Trained in {stats.metrics['train_runtime'] / 60:.0f} min. Adapter saved to {RUN_DIR / 'lora'}")

In [ ]:
# 6. Score on held-out songs
import time

FastVisionModel.for_inference(model)
chat = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': PROMPT}]}],
    add_generation_prompt=True)


def read_image(image):
    """One RGB bar image → the model's text answer."""
    inputs = tokenizer(image, chat, add_special_tokens=False, return_tensors='pt').to('cuda')
    output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def read_bar(path, scale=IMAGE_SCALE):
    return read_image(load_image(path, scale).convert('RGB'))


def placed(notes):
    return [(note['position'], tuple(sorted(note['drums'])), note['duration']) for note in notes]


def events(notes):
    """Ordered (drums, duration) hits: the baseline's sequence-accuracy definition."""
    return [hit[1:] for hit in placed(notes)]


def score(image, truth, scale=IMAGE_SCALE):
    text = read_bar(image, scale)
    try:
        notes = decode_text(text)
    except ValueError:
        notes = None  # unreadable output counts as a wrong bar
    return {
        'image': image, 'predicted': text, 'malformed': notes is None,
        'sequence_match': notes is not None and events(notes) == events(truth),
        'exact_match': notes is not None and placed(notes) == placed(truth),
    }


step = max(1, len(test_rows) // EVAL_BARS) if EVAL_BARS else 1
eval_rows = [] if (RUN_DIR / 'eval_results.json').exists() else test_rows[::step]
results, started = [], time.time()
for index, row in enumerate(eval_rows, 1):
    results.append({**score(row['image'], decode_text(row['target'])), 'target': row['target']})
    if index % 25 == 0:
        print(f"{index}/{len(eval_rows)}  sequence so far: {sum(r['sequence_match'] for r in results) / index:.1%}")

if not results:
    results = json.loads((RUN_DIR / 'eval_results.json').read_text())['bars']
    print('Held-out bars were already scored for this run:')
total = len(results)
# A third of the test bars have exactly the same notes as some training bar (13% are empty
# bars). Bars with notes the model never saw in training are the harder, more honest test.
seen = {row['target'] for row in train_rows}
new = [r for r in results if r['target'] not in seen]
summary = {
    'run': RUN_DIR.name, 'image_scale': IMAGE_SCALE,
    'bars_scored': total,
    'sequence_accuracy': sum(r['sequence_match'] for r in results) / total,
    'exact_with_positions': sum(r['exact_match'] for r in results) / total,
    'malformed_outputs': sum(r['malformed'] for r in results),
    'bars_with_unseen_notes': len(new),
    'sequence_accuracy_on_unseen_notes': sum(r['sequence_match'] for r in new) / max(1, len(new)),
}
if eval_rows:
    summary['seconds_per_bar'] = (time.time() - started) / total
    (RUN_DIR / 'eval_results.json').write_text(json.dumps({'summary': summary, 'bars': results}, indent=2))
print(json.dumps(summary, indent=2))
print('MobileNetV3 baseline: 12.5% on all 2,184 test bars, 12.8% on these 312 sampled bars')

In [ ]:
# 7. Look at the first few misses
for result in [r for r in results if not r['sequence_match']][:5]:
    display(Image.open(ROOT / result['image']))
    print('truth    :', result['target'])
    print('predicted:', result['predicted'], '\n')

In [ ]:
# 8. Score on the 45 bars DrumHub drew itself (eval/recognition/synthetic).
# The model has never seen this drawing style. Same "exact bars" rule as the benchmark lab,
# where GPT-5.6 Luna (high) got 51.1% and MobileNetV3 got 0%.
SYNTHETIC = 'eval/recognition/synthetic'
items = json.loads((ROOT / SYNTHETIC / 'manifest.json').read_text())['items']
# These images are 660x360, already about the size of an enlarged training bar, so no enlarging.
app_results = [{**score(f"{SYNTHETIC}/{item['image']}", item['notes'], scale=1), 'id': item['id'],
                'truth': item['notes']} for item in items]
app_summary = {
    'run': RUN_DIR.name, 'bars_scored': len(app_results),
    'exact_bars': sum(r['exact_match'] for r in app_results) / len(app_results),
    'sequence_accuracy': sum(r['sequence_match'] for r in app_results) / len(app_results),
    'malformed_outputs': sum(r['malformed'] for r in app_results),
}
(RUN_DIR / 'synthetic_results.json').write_text(json.dumps({'summary': app_summary, 'bars': app_results}, indent=2))
print(json.dumps(app_summary, indent=2))
for result in [r for r in app_results if not r['exact_match']][:5]:
    display(Image.open(ROOT / result['image']))
    print('truth    :', ' '.join(f"{n['position']}:{n['duration']}:{'+'.join(n['drums'])}" for n in result['truth']))
    print('predicted:', result['predicted'], '\n')

## Use this model in the DrumHub app

The next cell turns this Colab session into a temporary recognition server. It prints two
lines to paste into the `.env` file in the DrumHub project folder; then restart the app
(`npm start`) and import a bar as usual. The server stops when this session ends, and each
new session prints a new address and token.

In [ ]:
# 9. Serve the model to the DrumHub app while this session stays open
%pip install -q fastapi uvicorn python-multipart
import io, os, re, secrets, subprocess, threading
import uvicorn
from fastapi import FastAPI, File, Header, UploadFile
from fastapi.responses import JSONResponse

TOKEN = secrets.token_urlsafe(24)
TRAIN_HEIGHT = 146 * IMAGE_SCALE  # training bars were 146 px tall before enlarging
MAX_WIDTH = 2400
MAX_UPLOAD = 10 * 1024 * 1024
gpu = threading.Lock()            # one bar at a time on the GPU
server = FastAPI()


def failure(status, code, message):
    return JSONResponse({'error': {'code': code, 'message': message}}, status_code=status)


@server.get('/health')
def health():
    return {'status': 'ok', 'model': RUN_DIR.name, 'grid_slots': 32}


@server.post('/predict')
def predict(image: UploadFile = File(...), authorization: str = Header('')):
    if not secrets.compare_digest(authorization.encode(), f'Bearer {TOKEN}'.encode()):
        return failure(401, 'unauthorized', 'Wrong or missing token')
    data = image.file.read(MAX_UPLOAD + 1)
    if len(data) > MAX_UPLOAD:
        return failure(413, 'file_too_large', 'The image is larger than 10 MB')
    try:
        bar = Image.open(io.BytesIO(data)).convert('RGB')
    except Exception:
        return failure(415, 'unsupported_file', 'The file is not a PNG or JPEG image')
    # ponytail: assumes a crop about as tight as the training crops. If loosely cropped bars
    # read badly, find the staff lines and scale by their spacing instead of image height.
    width = min(MAX_WIDTH, max(1, round(bar.width * TRAIN_HEIGHT / bar.height)))
    with gpu:
        text = read_image(bar.resize((width, TRAIN_HEIGHT), Image.LANCZOS))
    try:
        notes = decode_text(text)
    except ValueError:
        return failure(422, 'unreadable_answer', 'The model could not read this bar')
    return {'notes': notes, 'model': RUN_DIR.name}


threading.Thread(target=lambda: uvicorn.run(server, host='127.0.0.1', port=8000, log_level='warning'),
                 daemon=True).start()

# A Cloudflare quick tunnel gives the local server a temporary public https address.
subprocess.run(['wget', '-q', '-O', 'cloudflared',
                'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
os.chmod('cloudflared', 0o755)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
                          stderr=subprocess.PIPE, text=True)
address = None
for line in tunnel.stderr:
    found = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if found:
        address = found.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()  # keep its log drained

if address:
    print('Paste these two lines into the .env file in the DrumHub project folder, then restart the app:\n')
    print(f'QWEN_OMR_URL={address}')
    print(f'QWEN_OMR_TOKEN={TOKEN}')
    print('\nThe token is a password for this server. Do not share it or commit it.')
else:
    print('The tunnel did not start. Run this cell again.')